# Task 3: Feature Engineering & RFM Extraction
In this section, we construct a high-yield feature engineering pipeline. We aggregate the transactional raw data at the `CustomerId` level to isolate Recency, Frequency, Monetary value, and transaction variance metrics.

In [1]:
import os
import pandas as pd
import numpy as np

# Safe path checking
data_path = "../data/raw/data.csv"
if not os.path.exists(data_path):
    data_path = "data/raw/data.csv"

df = pd.read_csv(data_path)

# Convert TransactionStartTime to a true datetime object object safely
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

# Establish a baseline date (1 day after the final transaction in the dataset) to calculate recency
baseline_date = df['TransactionStartTime'].max() + pd.Timedelta(days=1)

print(f"Data ready for aggregation. Operational baseline date set to: {baseline_date}")

Data ready for aggregation. Operational baseline date set to: 2019-02-14 10:01:28+00:00


In [2]:
# Group by CustomerId to extract behavioral features
customer_features = df.groupby('CustomerId').agg(
    # Recency: Days between baseline date and the most recent transaction
    Recency=('TransactionStartTime', lambda x: (baseline_date - x.max()).days),
    
    # Frequency: Total number of transactions executed by the customer
    Frequency=('TransactionId', 'count'),
    
    # Monetary Metrics
    Total_Spending=('Value', 'sum'),
    Average_Transaction_Value=('Value', 'mean'),
    Max_Transaction_Value=('Value', 'max'),
    
    # Variability Metrics
    Transaction_Value_Std=('Value', 'std'),
    
    # Target proxy indicator baseline
    Platform_Fraud_Occurrences=('FraudResult', 'sum')
).reset_index()

# Handle any missing standard deviations for customers with only 1 transaction
customer_features['Transaction_Value_Std'] = customer_features['Transaction_Value_Std'].fillna(0)

print(f"Feature engineering complete! Shape of customer profile matrix: {customer_features.shape}")
customer_features.head()

Feature engineering complete! Shape of customer profile matrix: (3742, 8)


,CustomerId,Recency,Frequency,Total_Spending,Average_Transaction_Value,Max_Transaction_Value,Transaction_Value_Std,Platform_Fraud_Occurrences
0,CustomerId_1,84,1,10000,10000.000000,10000,0.000000,0
1,CustomerId_10,84,1,10000,10000.000000,10000,0.000000,0
2,CustomerId_1001,90,5,30400,6080.000000,10000,4100.243895,0
3,CustomerId_1002,26,11,4775,434.090909,1500,518.805446,0
4,CustomerId_1003,12,6,32000,5333.333333,10000,3945.461528,0


In [3]:
# Establish simple percentile thresholds for our proxy rules
# High risk: Low frequency (bottom 25%) and low total monetary value (bottom 25%)
freq_threshold = customer_features['Frequency'].quantile(0.25)
monetary_threshold = customer_features['Total_Spending'].quantile(0.25)

def assign_proxy_risk_label(row):
    # Rule 1: Prioritize explicit historical platform issues
    if row['Platform_Fraud_Occurrences'] > 0:
        return 1 # Bad credit risk
    
    # Rule 2: Inactive, low-value spending behavior proxy
    if row['Frequency'] <= freq_threshold and row['Total_Spending'] <= monetary_threshold:
        return 1 # Bad credit risk
        
    return 0 # Good credit risk (Safe to offer BNPL loan)

# Apply our classification rules to build the target column
customer_features['Default_Proxy'] = customer_features.apply(assign_proxy_risk_label, axis=1)

print("--- Class Distribution of Generated Default Proxy Label ---")
print(customer_features['Default_Proxy'].value_counts())
print("\nPercentage Split:")
print(customer_features['Default_Proxy'].value_counts(normalize=True) * 100)

--- Class Distribution of Generated Default Proxy Label ---
Default_Proxy
0    3075
1     667
Name: count, dtype: int64

Percentage Split:
Default_Proxy
0    82.175307
1    17.824693
Name: proportion, dtype: float64


In [4]:
# Create directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# Save to processed directory
output_path = "../data/processed/customer_features.csv"
try:
    customer_features.to_csv(output_path, index=False)
except FileNotFoundError:
    output_path = "data/processed/customer_features.csv"
    customer_features.to_csv(output_path, index=False)

print(f"Pristine feature matrix successfully saved to: {output_path}")

Pristine feature matrix successfully saved to: ../data/processed/customer_features.csv
